# Week 2 - Day 1: Text-Cleaning Tests

This notebook tests the reusable functions in `src/text_cleaning.py`.

In [1]:
from pathlib import Path
import sys

current_directory = Path.cwd().resolve()
project_root = current_directory.parent if current_directory.name == 'notebooks' else current_directory
src_directory = project_root / 'src'

if str(src_directory) not in sys.path:
    sys.path.insert(0, str(src_directory))

from text_cleaning import (
    handle_missing_text,
    normalize_unicode,
    normalize_whitespace,
    replace_urls,
    normalize_hashtags,
    limit_repeated_punctuation,
    clean_text,
    clean_text_collection,
    remove_exact_duplicates,
    read_text_file,
    write_text_file,
)

print('Imported text_cleaning successfully.')

Imported text_cleaning successfully.


## 1. Test individual functions

In [2]:
assert handle_missing_text(None) == ''
assert handle_missing_text(123) == '123'
assert normalize_unicode('বাংলা') == 'বাংলা'
assert normalize_whitespace('এইমাত্র!!!     বড় খবর') == 'এইমাত্র!!! বড় খবর'
assert normalize_whitespace('  বাংলা\t  English\n text  ') == 'বাংলা English text'
assert replace_urls('বিস্তারিত: https://example.com/news') == 'বিস্তারিত: <URL>'
assert normalize_hashtags('#BreakingNews বড় খবর') == '<HASHTAG> BreakingNews বড় খবর'
assert limit_repeated_punctuation('কী?????? সত্যি!!!!!!') == 'কী??? সত্যি!!!'
assert clean_text(None) == ''

print('Individual function tests passed.')

Individual function tests passed.


## 2. Test the complete cleaning pipeline

In [3]:
raw_example = '#BreakingNews    বিস্তারিত: https://example.com!!!!!! 😲'
expected_example = '<HASHTAG> BreakingNews বিস্তারিত: <URL> !!! 😲'
actual_example = clean_text(raw_example)

print('Raw:     ', raw_example)
print('Expected:', expected_example)
print('Actual:  ', actual_example)

assert actual_example == expected_example
print('Complete pipeline test passed.')

Raw:      #BreakingNews    বিস্তারিত: https://example.com!!!!!! 😲
Expected: <HASHTAG> BreakingNews বিস্তারিত: <URL> !!! 😲
Actual:   <HASHTAG> BreakingNews বিস্তারিত: <URL> !!! 😲
Complete pipeline test passed.


## 3. Test collection cleaning

In [4]:
test_texts = [
    'এইমাত্র!!!     বড় খবর',
    None,
    '#BreakingNews বড় খবর 😲',
    'বিস্তারিত: https://example.com/news',
    'কী?????? সত্যি!!!!!!',
]

expected_texts = [
    'এইমাত্র!!! বড় খবর',
    '',
    '<HASHTAG> BreakingNews বড় খবর 😲',
    'বিস্তারিত: <URL>',
    'কী??? সত্যি!!!',
]

cleaned_texts = clean_text_collection(test_texts)
assert cleaned_texts == expected_texts

for raw, cleaned in zip(test_texts, cleaned_texts):
    print(f'{raw!r} -> {cleaned!r}')

print('Collection test passed.')

'এইমাত্র!!!     বড় খবর' -> 'এইমাত্র!!! বড় খবর'
None -> ''
'#BreakingNews বড় খবর 😲' -> '<HASHTAG> BreakingNews বড় খবর 😲'
'বিস্তারিত: https://example.com/news' -> 'বিস্তারিত: <URL>'
'কী?????? সত্যি!!!!!!' -> 'কী??? সত্যি!!!'
Collection test passed.


## 4. Test order-preserving duplicate removal

In [5]:
duplicate_texts = [
    'বড় খবর',
    'সরাসরি',
    'বড় খবর',
    'গুরুত্বপূর্ণ',
    'সরাসরি',
]

expected_unique_texts = ['বড় খবর', 'সরাসরি', 'গুরুত্বপূর্ণ']
unique_texts = remove_exact_duplicates(duplicate_texts)

assert unique_texts == expected_unique_texts
print(unique_texts)
print('Duplicate-removal test passed.')

['বড় খবর', 'সরাসরি', 'গুরুত্বপূর্ণ']
Duplicate-removal test passed.


## 5. Test file reading and writing

In [6]:
data_directory = project_root / 'data'
input_path = data_directory / 'sample_raw_texts.txt'
output_path = data_directory / 'sample_cleaned_texts.txt'

raw_texts = read_text_file(input_path)

if not raw_texts:
    raise FileNotFoundError(f'No sample texts found at {input_path}')

cleaned_texts = clean_text_collection(raw_texts)
unique_cleaned_texts = remove_exact_duplicates(cleaned_texts)
write_succeeded = write_text_file(unique_cleaned_texts, output_path)

assert write_succeeded
assert read_text_file(output_path) == unique_cleaned_texts

print(f'Cleaned file created: {output_path}')
print('File reading and writing tests passed.')

Cleaned file created: F:\GenAI\genai-learning\project_01_classical_baseline\data\sample_cleaned_texts.txt
File reading and writing tests passed.


## 6. Final test result

In [7]:
print('All tests passed successfully!')

All tests passed successfully!
